In [7]:
import os
import torch
from tqdm.notebook import tqdm
from dataclasses import dataclass, field
from typing import Optional
from datasets import load_dataset, load_from_disk
from peft import LoraConfig, prepare_model_for_kbit_training  # FP16 is fine
from transformers import AutoModelForCausalLM, AutoTokenizer, HfArgumentParser, TrainingArguments, DataCollatorForLanguageModeling
from trl import SFTTrainer
from huggingface_hub import interpreter_login


C:\Users\ankus\AppData\Local\Temp\ipykernel_13912\1947907620.py:9: FutureWarning: Support for Python 3.9 will be dropped in the next release (after its end-of-life on October 31, 2025). Please upgrade to Python 3.10 or newer.
  from trl import SFTTrainer


In [8]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))


True
1
NVIDIA GeForce RTX 2050


In [9]:
from huggingface_hub import interpreter_login

In [10]:
interpreter_login() 


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|



Enter your token (input will not be visible):  ········
Add token as git credential? (Y/n)  y


In [11]:
dataset = load_dataset("Amod/mental_health_counseling_conversations" , split='train')

In [12]:
dataset

Dataset({
    features: ['Context', 'Response'],
    num_rows: 3512
})

In [13]:
import pandas as pd
data = pd.DataFrame(dataset)

In [14]:
data.head(2)

,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."


In [15]:
#formatted
def formate_row(row):
    question = row["Context"]
    answer = row["Response"]
    formatted_data = f"[INST] {question} [/INST] {answer}"
    return formatted_data

In [16]:
data["Formatted"] = data.apply(formate_row , axis=1)

In [17]:
data

,Context,Response,Formatted
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb...",[INST] I'm going through some things with my f...
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see...",[INST] I'm going through some things with my f...
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...,[INST] I'm going through some things with my f...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...,[INST] I'm going through some things with my f...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...,[INST] I'm going through some things with my f...
...,...,...,...
3507,My grandson's step-mother sends him to school ...,Absolutely not! It is never in a child's best ...,[INST] My grandson's step-mother sends him to ...
3508,My boyfriend is in recovery from drug addictio...,I'm sorry you have tension between you and you...,[INST] My boyfriend is in recovery from drug a...
3509,The birth mother attempted suicide several tim...,"The true answer is, ""no one can really say wit...",[INST] The birth mother attempted suicide seve...
3510,I think adult life is making him depressed and...,How do you help yourself to believe you requir...,[INST] I think adult life is making him depres...


In [18]:
new_df = data.rename(columns={"Formatted": "Text"})

In [19]:
new_df

,Context,Response,Text
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb...",[INST] I'm going through some things with my f...
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see...",[INST] I'm going through some things with my f...
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...,[INST] I'm going through some things with my f...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...,[INST] I'm going through some things with my f...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...,[INST] I'm going through some things with my f...
...,...,...,...
3507,My grandson's step-mother sends him to school ...,Absolutely not! It is never in a child's best ...,[INST] My grandson's step-mother sends him to ...
3508,My boyfriend is in recovery from drug addictio...,I'm sorry you have tension between you and you...,[INST] My boyfriend is in recovery from drug a...
3509,The birth mother attempted suicide several tim...,"The true answer is, ""no one can really say wit...",[INST] The birth mother attempted suicide seve...
3510,I think adult life is making him depressed and...,How do you help yourself to believe you requir...,[INST] I think adult life is making him depres...


In [20]:
new_df = new_df[["Text"]]

In [21]:
new_df

,Text
0,[INST] I'm going through some things with my f...
1,[INST] I'm going through some things with my f...
2,[INST] I'm going through some things with my f...
3,[INST] I'm going through some things with my f...
4,[INST] I'm going through some things with my f...
...,...
3507,[INST] My grandson's step-mother sends him to ...
3508,[INST] My boyfriend is in recovery from drug a...
3509,[INST] The birth mother attempted suicide seve...
3510,[INST] I think adult life is making him depres...


## NOW FINE-TUNNING PHI-2 MODEL

In [22]:
new_df.to_csv("formatted_data.csv", index = False)

In [23]:
final_df = pd.read_csv("formatted_data.csv")

In [24]:
training_dataset = load_dataset(
    "csv",
    data_files="formatted_data.csv",  
    split="train"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [25]:
training_dataset

Dataset({
    features: ['Text'],
    num_rows: 3512
})

In [26]:
base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
new_model = "phi-2-mental-health"

In [27]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model,
                                         use_fast = True)

In [28]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [30]:
from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(load_in_4bit=True,
                                bnb_4bit_compute_dtype=torch.bfloat16,
                                bnb_4bit_quant_type='nf4',
                                bnb_4bit_use_double_quant=True,)

In [31]:
model =AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config = bnb_config
)

In [32]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())

True
1


In [33]:
model.config.use_cache = False
model.config.pretraining_tp = 1

In [34]:
model = prepare_model_for_kbit_training(model , use_gradient_checkpointing=True)

In [36]:
training_arguments = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=32,
    eval_steps=2000,
    logging_steps=15,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    save_steps=2000,
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_steps=-1)

In [37]:
peft_config = LoraConfig(
    r = 32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules = ["o_proj","q_proj","v_proj","k_proj","gate_proj","up_proj","down_proj"])

In [44]:
training_dataset = training_dataset.rename_column("Text", "text")

In [45]:
trainer = SFTTrainer(
    model=model,
    train_dataset=training_dataset,
    peft_config=peft_config,
    args=training_arguments,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

Adding EOS to train dataset:   0%|          | 0/3512 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3512 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (25993 > 2048). Running this sequence through the model will result in indexing errors


Truncating train dataset:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [46]:
trainer

In [47]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

  ········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\ankus\_netrc
wandb: Currently logged in as: ankush654123 (ankush654123-fdgfhgjhb-n) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


C:\Users\ankus\anaconda3\envs\new_env\lib\site-packages\torch\_dynamo\eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
15,2.385700
30,2.238600
45,2.156300
60,2.097600
75,1.991900
90,1.940500
105,1.907800
120,1.814600
135,1.757500
150,1.759900


TrainOutput(global_step=165, training_loss=1.9833553660999645, metrics={'train_runtime': 11356.9618, 'train_samples_per_second': 0.928, 'train_steps_per_second': 0.015, 'total_flos': 2.753054013657907e+16, 'train_loss': 1.9833553660999645, 'epoch': 3.0})

In [48]:
trainer.model.save_pretrained("./results_adapter")
tokenizer.save_pretrained("./results_adapter")

('./results_adapter\\tokenizer_config.json',
 './results_adapter\\special_tokens_map.json',
 './results_adapter\\chat_template.jinja',
 './results_adapter\\tokenizer.model',
 './results_adapter\\added_tokens.json',
 './results_adapter\\tokenizer.json')

In [53]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig, LlamaTokenizer
from peft import PeftModel

local_path = "results_adapter"

tokenizer =  AutoTokenizer.from_pretrained(local_path)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

device_target = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    local_path,
    quantization_config=bnb_config,
    device_map={"": device_target},  
    trust_remote_code=True,
    torch_dtype=torch.float16
)

`torch_dtype` is deprecated! Use `dtype` instead!


In [54]:
user_question = "I am not able to sleep in night. Do you have any suggestions?"

eval_prompt = f"""Answer the following question **first**. You may provide more relevant facts after the answer, but always start by directly answering the question.

Q: {user_question}
A:"""

inputs = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1
    )
    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True)

print("\n Model's Answer:\n", answer)

torch.cuda.empty_cache()

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



 Model's Answer:
 Sleeping is a basic human need and it is essential for our physical and mental health. There are many things that can affect your ability to sleep. If you are experiencing any of these symptoms, talk with your primary care physician or a local psychiatrist. Sometimes medication can help, but if this does not work, there are other things you can try. Here are some ideas:

1. Exercise regularly. Exercise releases endorphins which are natural mood boosters.
2. Avoid caffeine and alcohol before bedtime. Caffeine and alcohol interfere with sleep.
3. Use relaxing music or aromatherapy to help you fall asleep.
4. Create a relaxing environment. Turn off the TV, close the curtains, and turn off the lights.
5. Avoid screens at least 30 minutes before going to bed.
6. Avoid eating large meals close to bedtime. Eating late at night can disrupt your digestion and make it harder to sleep.
7. Avoid smoking and drinking alcohol close to bedtime. These substances can impair your
